In [ ]:
from sqlalchemy import create_engine

db_params = {
    'host': 'localhost',
    'database': 'postgres',
    'user': 'postgres',
    'password': 'postgres',
    'port': 5432
}

conn = create_engine(f"postgresql://{db_params['user']}:{db_params['password']}@{db_params['host']}:{db_params['port']}/{db_params['database']}")

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_sql_query('SELECT * FROM mutation_results_by_project_variant', conn)

# Define column names
col_dot_percent = 'Detected of Total (%)'
col_initial_dot_percent = 'Initial D.O.T. (%)'
col_dot_improvement = 'D.O.T. Improvement'

col_doc_percent = 'Detected of Covered (%)'
col_initial_doc_percent = 'Initial D.O.C. (%)'
col_doc_improvement = 'D.O.C. Improvement'

# Calculate detection percentages
# We can use the detected_pct from the view instead of calculating
df[col_dot_percent] = df['detected_pct']
df[col_doc_percent] = np.round((df['detected'] / df['covered']) * 100, 2)

# Create a dataframe with just the INITIAL values
initial_df = df[df['variant'] == 'INITIAL'].copy()
initial_df = initial_df[['project_id', col_dot_percent, col_doc_percent]]
initial_df = initial_df.rename(columns={
    col_dot_percent: col_initial_dot_percent,
    col_doc_percent: col_initial_doc_percent
})

# Merge back into the original dataframe
df = pd.merge(df, initial_df, on='project_id', how='left')

# Calculate improvements compared to INITIAL
mask = ~df['variant'].isin(['INITIAL', 'ORIGINAL'])
df[col_dot_improvement] = np.nan
df[col_doc_improvement] = np.nan
df.loc[mask, col_dot_improvement] = df.loc[mask, col_dot_percent] - df.loc[mask, col_initial_dot_percent]
df.loc[mask, col_doc_improvement] = df.loc[mask, col_doc_percent] - df.loc[mask, col_initial_doc_percent]
df = df.drop([col_initial_dot_percent, col_initial_doc_percent], axis=1)

# Select only the columns we want to display
display_columns = [
    'project_id', 'project_name', 'variant', 
    'total', 'covered', 'uncovered', 'survived', 'detected', 'killed', 'timed_out', 'memory_error',
    col_dot_percent, col_dot_improvement, col_doc_percent, col_doc_improvement
]

# Display the dataframe with only the selected columns
df[display_columns]

In [ ]:
df = pd.read_sql_query('SELECT *, variant_order(variant) as variant_sort_order FROM mutation_results_by_project_variant_mutator', conn)

percentage_columns = ['covered', 'uncovered', 'survived', 'detected', 'killed', 'timed_out', 'memory_error']
for col in percentage_columns:
    df[f'{col} (%)'] = (df[col] / df['total'] * 100).round(2)

# Step 2: Create a summary dataframe aggregated by project and variant
# Sum up the counts for each project+variant combination
summary_df = df.groupby(['project_id', 'project_name', 'variant', 'variant_sort_order']).agg({
    'total': 'sum',
    'covered': 'sum',
    'uncovered': 'sum',
    'survived': 'sum',
    'detected': 'sum',
    'killed': 'sum',
    'timed_out': 'sum',
    'memory_error': 'sum'
}).reset_index()

# Calculate percentages for the summary
for col in percentage_columns:
    summary_df[f'{col} (%)'] = (summary_df[col] / summary_df['total'] * 100).round(2)

# Step 3: Calculate differences compared to 'INITIAL' variant
# First, identify all unique project IDs
projects = summary_df['project_id'].unique()

# Create a list to store the results
diff_rows = []

for project_id in projects:
    project_data = summary_df[summary_df['project_id'] == project_id]
    
    baseline_variant = 'INITIAL'
    
    # Get the baseline percentages
    baseline = project_data[project_data['variant'] == baseline_variant].iloc[0]

    # For each variant in this project
    for _, row in project_data.iterrows():
        diff_row = row.copy()

        # Calculate differences in percentages
        for col in percentage_columns:
            if row['variant'] == baseline_variant:
                # For the baseline variant itself, set differences to NaN
                diff_row[f'{col} (%) Diff.'] = np.nan
            else:
                # Calculate the percentage point difference
                diff_row[f'{col} (%) Diff.'] = (row[f'{col} (%)'] - baseline[f'{col} (%)']).round(2)

        diff_rows.append(diff_row)

# Create the final dataframe with differences
result_df = pd.DataFrame(diff_rows)

# Organize columns: first the identifiers, then all percentage columns, then all diff columns
columns_to_keep = ['project_id', 'project_name', 'variant', 'variant_sort_order', 'total']

# Add all percentage columns
for col in percentage_columns:
    columns_to_keep.append(f'{col} (%)')

# Add all difference columns
for col in percentage_columns:
    columns_to_keep.append(f'{col} (%) Diff.')

result_df = result_df[columns_to_keep]

# Sort by project_id and variant_sort_order
result_df = result_df.sort_values(['project_id', 'variant_sort_order'])

# Drop the variant_sort_order column if you don't want it in the final output
if 'variant_sort_order' in result_df.columns:
    result_df = result_df.drop('variant_sort_order', axis=1)

# Display the results
result_df


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

def fetch_mutation_data(conn, variants=None):
    """Fetch and prepare mutation data from database."""
    query = "SELECT * FROM mutation_results_by_project_variant_mutator"
    if variants:
        variant_list = "', '".join(variants)
        query += f" WHERE variant IN ('{variant_list}')"

    results = pd.read_sql_query(query, conn)

    # Rename columns to match visualization expectations
    return results.rename(columns={
        'detected_pct': 'detection_rate',
        'detected_pct_diff': 'improvement'
    })

def get_sorted_variants(mutator_results, conn):
    """Get variants sorted by their order from database."""
    all_variants = sorted(
        mutator_results['variant'].unique(), 
        key=lambda v: pd.read_sql_query(f"SELECT variant_order('{v}')", conn).iloc[0, 0]
    )
    improvement_variants = [v for v in all_variants if v not in ['ORIGINAL', 'INITIAL', 'BASELINE']]
    return all_variants, improvement_variants

def create_variant_color_mapping(variants):
    """Create a consistent color mapping for variants using seaborn's colorblind palette."""
    # Use seaborn's colorblind-friendly palette
    color_palette = sns.color_palette("colorblind", n_colors=len(variants))

    # Convert RGB tuples to hex codes using matplotlib's color conversion
    hex_palette = [mcolors.to_hex(color) for color in color_palette]

    return {variant: hex_palette[i] for i, variant in enumerate(variants)}

def calculate_y_axis_limits(mutator_results, project_ids):
    """Calculate consistent y-axis limits across all plots."""
    detection_y_max = 0
    improvement_y_max = 0
    improvement_y_min = 0

    for project_id in project_ids:
        project_data = mutator_results[mutator_results['project_id'] == project_id]
        detection_y_max = max(detection_y_max, project_data['detection_rate'].max() * 1.1)

        project_improvements = project_data[project_data['improvement'].notna()]['improvement']
        if not project_improvements.empty:
            improvement_y_max = max(improvement_y_max, project_improvements.max() * 1.1)
            improvement_y_min = min(improvement_y_min, project_improvements.min() * 1.1)

    return detection_y_max, improvement_y_max, improvement_y_min

def plot_detection_rates(ax, project_data, all_variants, all_mutators, variant_colors):
    """Plot detection rate bars for a project."""
    mutator_positions = {mutator: i for i, mutator in enumerate(all_mutators)}
    width = 0.8 / len(all_variants)
    total_width = width * len(all_variants)
    group_offsets = -total_width/2 + width/2

    for variant_idx, variant in enumerate(all_variants):
        variant_data = project_data[project_data['variant'] == variant]
        detection_rates = {row['mutator']: row['detection_rate'] for _, row in variant_data.iterrows()}

        for mutator, pos in mutator_positions.items():
            rate = detection_rates.get(mutator, 0)
            if rate > 0:
                offset = group_offsets + width * variant_idx
                ax.bar(pos + offset, rate, width, color=variant_colors[variant])

def plot_improvements(ax, project_data, improvement_variants, all_mutators, variant_colors):
    """Plot improvement bars for a project."""
    if not improvement_variants:
        return

    mutator_positions = {mutator: i for i, mutator in enumerate(all_mutators)}
    width = 0.8 / len(improvement_variants)
    total_width = width * len(improvement_variants)
    group_offsets = -total_width/2 + width/2

    for variant_idx, variant in enumerate(improvement_variants):
        variant_data = project_data[project_data['variant'] == variant]
        improvements = {row['mutator']: row['improvement'] 
                       for _, row in variant_data.iterrows() 
                       if row['improvement'] is not None}

        for mutator, pos in mutator_positions.items():
            impr = improvements.get(mutator, 0)
            if impr != 0:
                offset = group_offsets + width * variant_idx
                ax.bar(pos + offset, impr, width, color=variant_colors[variant])

def configure_axis(ax, title, ylabel, x_min, x_max, y_min, y_max, all_mutators, show_xticklabels=False):
    """Configure axis properties."""
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(np.arange(len(all_mutators)))

    if show_xticklabels:
        ax.set_xticklabels(all_mutators, rotation=90, ha='center')
    else:
        ax.set_xticklabels([])

    ax.grid(axis='y', linestyle='--', alpha=0.7)

# Main visualization function
def visualize_mutation_results(conn, variants=None):
    """Create comprehensive visualization of mutation testing results."""
    # Set seaborn style for better aesthetics
    sns.set_style("whitegrid")

    # Fetch and prepare data
    mutator_results = fetch_mutation_data(conn, variants)
    all_variants, improvement_variants = get_sorted_variants(mutator_results, conn)
    all_mutators = sorted(mutator_results['mutator'].unique())
    project_ids = mutator_results['project_id'].unique()
    project_count = len(project_ids)

    # Calculate axis limits
    detection_y_max, improvement_y_max, improvement_y_min = calculate_y_axis_limits(
        mutator_results, project_ids)

    # Create color mapping using seaborn's colorblind palette
    variant_colors = create_variant_color_mapping(all_variants)

    # Create figure and grid
    fig = plt.figure(figsize=(18, 5 + 4 * project_count))
    gs = GridSpec(project_count, 2, width_ratios=[3, 2])

    # Set fixed x-axis limits
    x_min, x_max = -0.5, len(all_mutators) - 0.5

    # Create legend
    legend_handles = [plt.Rectangle((0, 0), 1, 1, color=variant_colors[variant]) 
                     for variant in all_variants]
    fig.legend(
        legend_handles, all_variants, loc='upper center',
        ncol=min(len(all_variants), 5), bbox_to_anchor=(0.5, 0.98), fontsize='small'
    )

    # Create plots for each project
    for i, project_id in enumerate(project_ids):
        project_data = mutator_results[mutator_results['project_id'] == project_id]
        if project_data.empty:
            continue

        # Get project name
        project_name = pd.read_sql_query(
            f"SELECT project_name({project_id})", conn).iloc[0, 0]

        # Create subplots
        ax1 = fig.add_subplot(gs[i, 0])  # Detection rate plot
        ax2 = fig.add_subplot(gs[i, 1])  # Improvement plot

        # Plot detection rates
        plot_detection_rates(ax1, project_data, all_variants, all_mutators, variant_colors)
        configure_axis(
            ax1, f'Detection Rate - Project ID: {project_id} - {project_name}',
            'Detection Rate (%)', x_min, x_max, 0, detection_y_max,
            all_mutators, show_xticklabels=(i == project_count - 1)
        )

        # Plot improvements
        plot_improvements(ax2, project_data, improvement_variants, all_mutators, variant_colors)
        configure_axis(
            ax2, f'Improvement - Project ID: {project_id} - {project_name}',
            'Improvement (%)', x_min, x_max, improvement_y_min, improvement_y_max,
            all_mutators, show_xticklabels=(i == project_count - 1)
        )
        ax2.axhline(y=0, color='k', linestyle='-', alpha=0.3)

    # Adjust layout
    plt.tight_layout(rect=[0, 0.03, 1, 0.92])
    plt.subplots_adjust(hspace=0.3, top=0.88)

    return fig

variants_to_plot = ['INITIAL', 'NAIVE_1000_TRIES', 'IMPROVED_1000_TRIES']
fig = visualize_mutation_results(conn, variants_to_plot)
plt.show()
